# Week 12 deliverables, Kaggle version
Same as the Colab version, but for **Kaggle** (kaggle.com), which gives about 30 free GPU hours a week.

**Before you start (one time):**
1. On the right, open **Session options**. Set **Accelerator** to **GPU T4 x2** (or P100).
2. Turn **Internet** on. Kaggle asks you to verify your phone number the first time; this is free.

Then click **Run All**. Results go to `/kaggle/working/results`.
When it finishes, click **Save Version > Save & Run All (Commit)**, or download the files from the **Output** panel on the right, so they are kept.

In [ ]:
import os
OUT = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)
print("Saving results to", OUT)

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn sentencepiece pandas

## Step 0. Settings

In [ ]:
import json, random, time, numpy as np, pandas as pd, torch
from datasets import load_dataset
from transformers import AutoTokenizer

LANGS = {"nya": "nya_Latn", "tum": "tum_Latn"}   # Chichewa, Tumbuka
SEEDS = [1, 2, 3, 4, 5]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)   # should say cuda

MODEL_CANDIDATES = {
    "xlmr-base":          ["xlm-roberta-base"],
    "afro-xlmr-base":     ["Davlan/afro-xlmr-base"],
    "afro-xlmr-base-76L": ["Davlan/afro-xlmr-base-76L", "Davlan/afro-xlmr-base-76L_script"],
}
def resolve(name):
    for repo in MODEL_CANDIDATES[name]:
        try:
            AutoTokenizer.from_pretrained(repo); print(f"{name}: using {repo}"); return repo
        except Exception as e:
            print(f"{name}: {repo} not found ({type(e).__name__}), trying next")
    raise RuntimeError(f"No working repo for {name}")
MODELS = {k: resolve(k) for k in MODEL_CANDIDATES}
json.dump(MODELS, open(f"{OUT}/models_used.json", "w"), indent=2)

## Step 1. Load SIB-200

In [ ]:
sib = {lang: load_dataset("Davlan/sib200", cfg) for lang, cfg in LANGS.items()}
for lang in LANGS: print(lang, {s: len(sib[lang][s]) for s in sib[lang]})
LABELS = sorted(set(sib["tum"]["train"]["category"]))
label2id = {l: i for i, l in enumerate(LABELS)}
print("Topics:", LABELS)

## Step 2. Tokens per word (deliverable 1)

In [ ]:
path = f"{OUT}/tokens_per_word.csv"
if os.path.exists(path):
    fert = pd.read_csv(path)
else:
    rows = []
    for mname, repo in MODELS.items():
        tok = AutoTokenizer.from_pretrained(repo)
        for lang in LANGS:
            texts = [t for s in ["train", "validation", "test"] for t in sib[lang][s]["text"]]
            nw = nt = ns = 0
            for t in texts:
                for w in t.split():
                    p = tok.tokenize(w); nw += 1; nt += len(p); ns += len(p) > 1
            rows.append({"model": mname, "lang": lang, "fertility": round(nt/nw, 3),
                         "split_words_%": round(100*ns/nw, 1)})
    fert = pd.DataFrame(rows); fert.to_csv(path, index=False)
fert

## Step 3. Corpus size in MB (deliverable 2)
Counts up to 200,000 documents per source and language. If the limit is reached, the size is written as "at least".

In [ ]:
LIMIT = 200_000
def count_mb(make_stream, key="text"):
    total = docs = 0; t0 = time.time()
    for ex in make_stream():
        total += len(ex[key].encode("utf-8")); docs += 1
        if docs >= LIMIT or time.time() - t0 > 600: break   # stop after 200k docs or 10 minutes
    return round(total/1e6, 2), docs

sources = {
    "Glot500-c": {"nya": lambda: load_dataset("cis-lmu/Glot500", "nya_Latn", split="train", streaming=True),
                  "tum": lambda: load_dataset("cis-lmu/Glot500", "tum_Latn", split="train", streaming=True)},
    "MADLAD-400 clean": {"nya": lambda: load_dataset("allenai/madlad-400", languages=["ny"], split="clean", streaming=True, trust_remote_code=True),
                         "tum": lambda: load_dataset("allenai/madlad-400", languages=["tum"], split="clean", streaming=True, trust_remote_code=True)},
}
path = f"{OUT}/corpus_size.csv"
if os.path.exists(path):
    corpus = pd.read_csv(path)
else:
    rows = []
    for src, d in sources.items():
        for lang in LANGS:
            try:
                mb, docs = count_mb(d[lang])
                rows.append({"source": src, "lang": lang, "MB": mb, "docs": docs,
                             "note": "at least (limit reached)" if docs >= LIMIT else ""})
            except Exception as e:
                rows.append({"source": src, "lang": lang, "MB": None, "docs": None,
                             "note": f"{type(e).__name__}: {str(e)[:120]}"})
            print(rows[-1])
    corpus = pd.DataFrame(rows); corpus.to_csv(path, index=False)
corpus

## Step 4. Fine-tune the baselines (deliverable 3)
20 runs (2 models x 2 languages x 5 seeds). Each run trains for 10 epochs, checks the dev set after every epoch, and keeps the test predictions from the best dev epoch. Nothing large is saved, so it is fast.
After each run a line like `afro-xlmr-base nya seed 1 -> test macro-F1 74.2 (best dev epoch 7)` is printed and saved to Drive.

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup, set_seed
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

def run(mname, lang, seed, epochs=10, lr=2e-5, bs=16, max_len=128):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODELS[mname])
    def prep(b):
        enc = tok(b["text"], truncation=True, max_length=max_len)
        enc["labels"] = [label2id[c] for c in b["category"]]
        return enc
    ds = sib[lang].map(prep, batched=True, remove_columns=sib[lang]["train"].column_names)
    coll = DataCollatorWithPadding(tok)
    train = DataLoader(ds["train"], batch_size=bs, shuffle=True, collate_fn=coll)
    dev = DataLoader(ds["validation"], batch_size=64, collate_fn=coll)
    test = DataLoader(ds["test"], batch_size=64, collate_fn=coll)
    model = AutoModelForSequenceClassification.from_pretrained(MODELS[mname], num_labels=len(LABELS)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.1*epochs*len(train)), epochs*len(train))
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    def predict(loader):
        model.eval(); P = []; Y = []
        with torch.no_grad():
            for b in loader:
                b = {k: v.to(DEVICE) for k, v in b.items()}
                with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                    P.append(model(**b).logits.argmax(-1).cpu()); Y.append(b["labels"].cpu())
        return torch.cat(P).numpy(), torch.cat(Y).numpy()
    best_dev, best_ep, best_test = -1, -1, None
    for ep in range(1, epochs + 1):
        model.train()
        for b in train:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                loss = model(**b).loss
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        p, y = predict(dev); f = f1_score(y, p, average="macro")
        if f > best_dev:
            best_dev, best_ep = f, ep
            best_test = predict(test)
    y_pred, y_gold = best_test
    np.save(f"{OUT}/pred_{mname}_{lang}_s{seed}.npy", y_pred)
    np.save(f"{OUT}/gold_{lang}.npy", y_gold)
    del model; torch.cuda.empty_cache()
    return round(100*f1_score(y_gold, y_pred, average="macro"), 2), round(100*best_dev, 2), best_ep

BASELINES = ["afro-xlmr-base", "afro-xlmr-base-76L"]
res_path = f"{OUT}/baselines.csv"
done = pd.read_csv(res_path) if os.path.exists(res_path) else pd.DataFrame(columns=["model","lang","seed","macro_f1","dev_f1","best_epoch"])
for mname in BASELINES:
    for lang in LANGS:
        for seed in SEEDS:
            if ((done.model == mname) & (done.lang == lang) & (done.seed == seed)).any():
                print("already done:", mname, lang, seed); continue
            t0 = time.time()
            f1, dev_f1, ep = run(mname, lang, seed)
            print(f"{mname} {lang} seed {seed} -> test macro-F1 {f1} (best dev epoch {ep}) [{(time.time()-t0)/60:.1f} min]")
            done = pd.concat([done, pd.DataFrame([{"model": mname, "lang": lang, "seed": seed,
                                                   "macro_f1": f1, "dev_f1": dev_f1, "best_epoch": ep}])])
            done.to_csv(res_path, index=False)

## Step 5. Baseline table

In [ ]:
base = pd.read_csv(f"{OUT}/baselines.csv")
table = base.groupby(["model", "lang"]).macro_f1.agg(["mean", "std", "count"]).round(2)
table.to_csv(f"{OUT}/baseline_table.csv")
table

## Step 6. Paired bootstrap over sentences
Resamples the same test sentences for both models 2,000 times. Shows the difference in macro-F1 (B minus A), its 95% interval, and how often B is not better. Predictions are combined over the 5 seeds by majority vote.

In [ ]:
from collections import Counter
def vote(mname, lang):
    P = np.stack([np.load(f"{OUT}/pred_{mname}_{lang}_s{s}.npy") for s in SEEDS])
    return np.array([Counter(c).most_common(1)[0][0] for c in P.T])
def paired_bootstrap(a, b, lang, n=2000, seed=0):
    gold = np.load(f"{OUT}/gold_{lang}.npy"); pa, pb = vote(a, lang), vote(b, lang)
    rng = np.random.default_rng(seed); diffs = []
    for _ in range(n):
        i = rng.integers(0, len(gold), len(gold))
        diffs.append(100*(f1_score(gold[i], pb[i], average="macro") - f1_score(gold[i], pa[i], average="macro")))
    d = np.array(diffs)
    return {"A": a, "B": b, "lang": lang, "n_test": len(gold), "mean_diff_B_minus_A": round(d.mean(), 2),
            "CI95_low": round(np.percentile(d, 2.5), 2), "CI95_high": round(np.percentile(d, 97.5), 2),
            "p_B_not_better": round((d <= 0).mean(), 3)}
boot = pd.DataFrame([paired_bootstrap("afro-xlmr-base", "afro-xlmr-base-76L", l) for l in LANGS])
boot.to_csv(f"{OUT}/bootstrap_base_vs_76L.csv", index=False)
boot

## Step 7. Show everything
All files are already in your Google Drive folder `term_project_results`. Take a screenshot of this output and send it to me.

In [ ]:
print(open(f"{OUT}/models_used.json").read())
print(fert.to_string()); print(); print(corpus.to_string()); print(); print(table.to_string()); print(); print(boot.to_string())